# Detección de Carga mediante Programación Genética (GP)

Este notebook implementa un modelo de **Programación Genética (GP)** utilizando la librería `deap` para resolver un problema de **regresión simbólica**. 

### Objetivo
Encontrar una fórmula matemática óptima que estime la **Carga** (peso transportado por el vehículo) a partir de las variables de telemetría dinámica:
* `Pila` (Nivel de batería actual)
* `DPila` (Tasa de cambio de batería / consumo)
* `Emisiones` (Emisiones acumuladas)
* `DEmisiones` (Tasa de cambio de emisiones)

Esto permite "detectar" de forma indirecta e inteligente la carga física que transporta el vehículo sin necesidad de sensores de peso directos.

In [ ]:
# Celda 1: Importación de librerías
import operator
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from deap import base, creator, tools, gp, algorithms

print("Librerías cargadas correctamente.")

## Carga y Preprocesamiento de Datos
Leemos el archivo histórico de rutas generado por la simulación de ArcGIS (`registros_arcgis_rutas.csv`).

In [ ]:
# Celda 2: Cargar y preparar el dataset
ruta_csv = 'registros_arcgis_rutas.csv'

try:
    df = pd.read_csv(ruta_csv)
    print(f"Dataset cargado con éxito. Total de registros: {len(df)}")
    
    # Seleccionamos las columnas relevantes para entrenar
    features = ['Pila', 'DPila', 'Emisiones', 'DEmisiones']
    target = 'Carga'
    
    # Eliminar filas con valores nulos si existen
    df_clean = df[features + [target]].dropna()
    
    # Dividir en entrenamiento (80%) y prueba (20%)
    df_train = df_clean.sample(frac=0.8, random_state=42)
    df_test = df_clean.drop(df_train.index)
    
    print(f"Registros de entrenamiento: {len(df_train)}")
    print(f"Registros de prueba: {len(df_test)}")
    
    # Mostrar las primeras filas del dataset de entrenamiento
    print("\nPrimeros registros de entrenamiento:")
    print(df_train.head())
    
except Exception as e:
    print(f"Error al cargar los datos: {e}")
    print("Generando datos sintéticos de respaldo para demostración...")
    
    # Respaldo: Generar datos sintéticos si el archivo no está accesible
    np.random.seed(42)
    n_samples = 500
    pila = np.random.uniform(20.0, 100.0, n_samples)
    carga = np.random.choice([15.0, 25.0, 40.0], n_samples)  # Rojo, Verde, Azul
    dpila = - (0.05 + carga * 0.005) + np.random.normal(0, 0.01, n_samples)
    emisiones = np.random.uniform(0.1, 5.0, n_samples)
    demisiones = (0.01 + carga * 0.001) + np.random.normal(0, 0.0005, n_samples)
    
    df_train = pd.DataFrame({
        'Pila': pila[:400], 'DPila': dpila[:400], 
        'Emisiones': emisiones[:400], 'DEmisiones': demisiones[:400],
        'Carga': carga[:400]
    })
    df_test = pd.DataFrame({
        'Pila': pila[400:], 'DPila': dpila[400:], 
        'Emisiones': emisiones[400:], 'DEmisiones': demisiones[400],
        'Carga': carga[400:]
    })
    print("Datos sintéticos creados con éxito.")

## Configuración del Conjunto de Primitivas y Terminales de GP
Definimos los operadores matemáticos permitidos en los árboles de ecuaciones y las variables que representan nuestras terminales.

In [ ]:
# Celda 3: Definición del conjunto de primitivas (pset)

# Operaciones seguras para evitar divisiones por cero, overflow y NaN
def div_segura(izq, der):
    return np.where(np.abs(der) > 1e-6, izq / der, 1.0)

def log_seguro(val):
    return np.log(np.clip(np.abs(val), 1e-6, None))

def exp_segura(val):
    return np.exp(np.clip(val, -10, 10))

# Definimos el PrimitiveSet con 4 variables de entrada
pset = gp.PrimitiveSet("MAIN", 4)
pset.addPrimitive(np.add, 2, name="add")
pset.addPrimitive(np.subtract, 2, name="sub")
pset.addPrimitive(np.multiply, 2, name="mul")
pset.addPrimitive(div_segura, 2, name="div")
pset.addPrimitive(np.sin, 1, name="sin")
pset.addPrimitive(np.cos, 1, name="cos")
pset.addPrimitive(exp_segura, 1, name="exp")
pset.addPrimitive(log_seguro, 1, name="log")

# Constantes efímeras entre -5 y 5 para calibración
def rand_const():
    return random.uniform(-5, 5)
pset.addEphemeralConstant("rand_const", rand_const)

# Renombrar argumentos para que correspondan con nuestras variables físicas
pset.renameArguments(ARG0='Pila')
pset.renameArguments(ARG1='DPila')
pset.renameArguments(ARG2='Emisiones')
pset.renameArguments(ARG3='DEmisiones')

print("Primitivas y terminales inicializadas correctamente.")

## Inicialización de Clases y Operaciones de DEAP
Configuramos la estructura del fitness (queremos minimizar el error de estimación) y registramos los operadores de mutación, cruce y selección.

In [ ]:
# Celda 4: Registro de operadores y funciones en la DEAP Toolbox

# Queremos MINIMIZAR el error absoluto/cuadrático de predicción
if not hasattr(creator, "FitnessMin"):
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
if not hasattr(creator, "Individual"):
    creator.create("Individual", gp.PrimitiveTree, fitness=creator.FitnessMin)

toolbox = base.Toolbox()

# Creador de árboles con profundidad inicial de entre 1 y 4 niveles
toolbox.register("expr", gp.genHalfAndHalf, pset=pset, min_=1, max_=4)
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.expr)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("compile", gp.compile, pset=pset)

# Selección por torneo
toolbox.register("select", tools.selTournament, tournsize=3)
# Cruce de un punto
toolbox.register("mate", gp.cxOnePoint)
# Mutación uniforme
toolbox.register("mutate", gp.mutUniform, expr=toolbox.expr, pset=pset)

# Limitar la altura de los árboles a 7 niveles para evitar el "bloat" (crecimiento excesivo y lento)
MAX_HEIGHT = 7
toolbox.decorate("mate", gp.staticLimit(key=operator.attrgetter("height"), max_value=MAX_HEIGHT))
toolbox.decorate("mutate", gp.staticLimit(key=operator.attrgetter("height"), max_value=MAX_HEIGHT))

print("Estructura evolutiva configurada.")

## Definición de la Función de Fitness (Error Absoluto Medio)
Evaluamos cada individuo calculando el **Mean Absolute Error (MAE)** de la ecuación simbólica frente al set de entrenamiento. Añadimos una penalización ligera basada en el tamaño del árbol para guiar al algoritmo hacia ecuaciones más simples y comprensibles.

In [ ]:
# Celda 5: Función de evaluación

def evaluar_carga(individuo, df_datos):
    try:
        # Compilar el árbol en una función ejecutable de Python
        funcion_modelo = toolbox.compile(expr=individuo)
        
        # Obtener variables de entrada vectorizadas
        p = df_datos['Pila'].values
        dp = df_datos['DPila'].values
        em = df_datos['Emisiones'].values
        dem = df_datos['DEmisiones'].values
        carga_real = df_datos['Carga'].values
        
        # Evaluar la función
        prediccion = funcion_modelo(p, dp, em, dem)
        
        # Si la salida es una constante, expandirla a un array
        if np.isscalar(prediccion):
            prediccion = np.full_like(carga_real, prediccion)
            
        # Calcular el Error Absoluto Medio (MAE)
        mae = np.mean(np.abs(prediccion - carga_real))
        
        # Si produce infinitos o NaN, penalizar fuertemente
        if not np.isfinite(mae):
            return (1e9,)
            
        # Penalización por complejidad del árbol (parsimonia)
        penalizacion = len(individuo) * 0.05
        
        return (mae + penalizacion,)
        
    except Exception:
        return (1e9,)

# Registrar la función de evaluación usando el set de entrenamiento
toolbox.register("evaluate", evaluar_carga, df_datos=df_train)
print("Función de evaluación registrada con éxito.")

## Ejecución del Proceso Evolutivo
Corremos el algoritmo genético para evolucionar la población a lo largo de varias generaciones.

In [ ]:
# Celda 6: Ejecución del algoritmo evolutivo

# Parámetros de evolución
TAM_POBLACION = 150
NUM_GENERACIONES = 40
PROB_CRUCE = 0.8
PROB_MUTACION = 0.15

# Inicializar población
pop = toolbox.population(n=TAM_POBLACION)
hof = tools.HallOfFame(1)

# Configurar estadísticas a registrar
stats = tools.Statistics(lambda ind: ind.fitness.values[0])
stats.register("Min", np.min)
stats.register("Avg", np.mean)

print("Iniciando evolución...")
pop, logbook = algorithms.eaSimple(
    pop, toolbox, 
    cxpb=PROB_CRUCE, 
    mutpb=PROB_MUTACION, 
    ngen=NUM_GENERACIONES, 
    stats=stats, 
    halloffame=hof, 
    verbose=True
)

print("\n¡Evolución completada!")

## Análisis de Resultados
Evaluamos el mejor modelo simbólico en los conjuntos de datos de entrenamiento y prueba, visualizamos la ecuación obtenida y graficamos su rendimiento.

In [ ]:
# Celda 7: Evaluación y visualización del mejor individuo
mejor_individuo = hof[0]
print("=== MEJOR ECUACIÓN SIMBÓLICA HALLADA ===")
print(mejor_individuo)
print(f"\nComplejidad (Nodos): {len(mejor_individuo)} | Altura: {mejor_individuo.height}")

# Compilar la mejor ecuación matemática
funcion_estimadora = toolbox.compile(expr=mejor_individuo)

# Evaluar en set de entrenamiento
y_train_pred = funcion_estimadora(
    df_train['Pila'].values, df_train['DPila'].values, 
    df_train['Emisiones'].values, df_train['DEmisiones'].values
)
mae_train = np.mean(np.abs(y_train_pred - df_train['Carga'].values))

# Evaluar en set de prueba (generalización)
y_test_pred = funcion_estimadora(
    df_test['Pila'].values, df_test['DPila'].values, 
    df_test['Emisiones'].values, df_test['DEmisiones'].values
)
mae_test = np.mean(np.abs(y_test_pred - df_test['Carga'].values))

print(f"\nError Absoluto Medio (MAE) Entrenamiento: {mae_train:.4f} kg")
print(f"Error Absoluto Medio (MAE) Generalización (Prueba): {mae_test:.4f} kg")

In [ ]:
# Celda 8: Gráficos de evolución y predicción
plt.figure(figsize=(15, 5))

# 1. Gráfico de Historial de Evolución
plt.subplot(1, 2, 1)
generaciones = logbook.select("gen")
min_fitness = logbook.select("Min")
avg_fitness = logbook.select("Avg")
plt.plot(generaciones, min_fitness, label="Mejor Fitness (Min Error)", color='crimson', lw=2)
plt.plot(generaciones, avg_fitness, label="Promedio Población", color='royalblue', linestyle='--')
plt.title("Progreso Evolutivo de Programación Genética")
plt.xlabel("Generación")
plt.ylabel("Fitness (MAE + Parsimonia)")
plt.grid(True, alpha=0.3)
plt.legend()

# 2. Gráfico Comparativo de Carga Real vs Predicción (en Test)
plt.subplot(1, 2, 2)
test_sort_indices = np.argsort(df_test['Carga'].values)
plt.plot(df_test['Carga'].values[test_sort_indices], 'go', label='Carga Real (CV)', alpha=0.6)
plt.plot(y_test_pred[test_sort_indices], 'r-', label='Predicción de GP', lw=2)
plt.title("Comparación: Carga Real vs Estimación de GP (Test)")
plt.xlabel("Muestras de Prueba (Ordenadas por peso)")
plt.ylabel("Peso / Carga (kg)")
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()